# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [58]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [59]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id'],
)
# TODO: assert the row count and the quantity total
assert len(lines) == sum(len(o['lines']) for o in orders_json) == 4
assert lines['qty'].sum() == sum(l['qty'] for o in orders_json for l in o['lines']) == 7

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [60]:
merged = lines_df.merge(
    pd.DataFrame(vendors_json),
    how='left',
    on='vendor_id',
    indicator=True,
)

unmatched = merged[merged['_merge'] == 'left_only']
unmatched_vendors = unmatched['vendor_id'].unique().tolist()
unmatched_units = unmatched['qty'].sum()

print(unmatched_vendors)
print(unmatched_units)

['V-07']
3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [61]:
# TODO
merged['unmatched_vendor'] = merged['_merge'].eq('left_only')
merged['name'] = merged['name'].fillna('UNMATCHED VENDOR')
merged['zone'] = merged['zone'].fillna('Unknown')
merged = merged.drop(columns='_merge')

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [62]:
# TODO
totals = merged.groupby('zone')['qty'].sum().sort_values(ascending=False)
print(totals)
print(totals.sum())

zone
A          4
Unknown    3
Name: qty, dtype: int64
7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [63]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
filled = [
    a if has_lines(a) else {**a, 'lines': [{'item': 'no lines'}]}
    for a in ragged_json
]

joined = pd.json_normalize(filled, record_path='lines', meta=['order', 'vendor_id'])
print(len(joined))

3


### Q6 — Validate

In [64]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

AssertionError: the vendor join changed the row count

### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) This shape is useful because a line item is the smallest unit in this dataset. The nested list is an easy way for the data to iterate through.

b) 0 is a defined fact; a quantity of 0 is a valid entry for 'qty.' A missing value indicates that there is a value that is not being filled out. This would give you faulty totals or make other values seem as though they have been calculated in their entirety when that is not the case.